---

Lab Jack

---

In [2]:
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl, pglLabJack, pglSettingsManager
pgl = pgl()

# close any existing windows
pgl.cleanUp()


================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl->pglBase:checkOS) Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
(pgl->pglBase:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.6.2
(pgl->pglBase:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl->pglBase:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl->pglBase:checkOS)    Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pgl->pglBase:checkOS)    LED Cinema Display: 1600 x 1000 (Unknown type) GammaTabl

In [ ]:
labjack = pglLabJack()

In [ ]:

pgl.labJackSettings()

In [ ]:
pgl.settings()

In [ ]:
pgl.settings()

In [5]:
# HARDWARE CHARACTERIZATION — run only with an idle, already-open pglLabJack.
# No stream starts/stops, pin writes, profile writes, or device closure.
# Artifacts are retained; nothing is automatically deleted.

import json
import math
from pathlib import Path
from uuid import uuid4

from pgl.pglTimestamp import pglTimestamp


# Preserve previous runs when this notebook cell is rerun.
if "labJackClockCharacterizationRuns" not in globals():
    labJackClockCharacterizationRuns = []


def characterizeLabJackClock(labJack, artifactDirectory):
    candidateTicksPerSecond = 40_000_000
    timerModulus = 1 << 32
    durationTargets = (0.050, 0.100, 0.250, 1.000)
    repeatCount = 3

    artifactDirectory = Path(artifactDirectory).expanduser().resolve()
    artifactDirectory.mkdir(parents=True, exist_ok=True)
    reportId = str(uuid4())
    artifactPath = artifactDirectory / f"labJack-clock-characterization-{reportId}.json"

    report = {
        "reportId": reportId,
        "method": "bracketedCoreTimerCharacterization-v1",
        "artifactPath": str(artifactPath),
        "candidateTicksPerSecond": candidateTicksPerSecond,
        "candidateWrapSeconds": timerModulus / candidateTicksPerSecond,
        "deviceIdentity": {
            "type": getattr(labJack, "type", None),
            "serialNumber": getattr(labJack, "serialNumber", None),
            "connectionType": getattr(labJack, "connectionType", None),
        },
        "assumptions": [
            "CORE_TIMER is a fresh uint32 timer sampled during each read transaction.",
            "40 MHz is a candidate frequency, not established by this test.",
            "Offset intersection assumes a constant nominal rate within each collection.",
            "One candidate timer tick expands each bracket for quantization.",
            "Host timestamp uncertainty is not quantified.",
            "Short-collection epoch resolution is conditional on the candidate timer model.",
        ],
        "registers": {},
        "collections": [],
        "status": "starting",
        "failure": None,
    }
    # Retain the report before the first hardware operation can fail.
    labJackClockCharacterizationRuns.append(report)

    def percentile(values, fraction):
        ordered = sorted(values)
        position = (len(ordered) - 1) * fraction
        lowerIndex = int(math.floor(position))
        upperIndex = int(math.ceil(position))
        weight = position - lowerIndex
        return ordered[lowerIndex] * (1 - weight) + ordered[upperIndex] * weight

    def summarizeCollection(collection):
        observations = collection["observations"]
        if len(observations) < 2:
            raise RuntimeError("Fewer than two observations; characterization inadequate.")

        first = observations[0]
        tickReference = first["rawTicks"]
        hostReferenceBase = first["hostMidpoint"]
        previousTicks = None

        for observation in observations:
            elapsedHost = observation["hostMidpoint"] - hostReferenceBase
            expectedTicks = tickReference + elapsedHost * candidateTicksPerSecond
            wrapNumber = round(
                (expectedTicks - observation["rawTicks"]) / timerModulus
            )
            resolvedTicks = observation["rawTicks"] + wrapNumber * timerModulus
            observation.update({
                "candidateWrapNumber": wrapNumber,
                "candidateUnwrappedTicks": resolvedTicks,
                "epochResolutionMethod": "nearestCandidateNominalPrediction",
                "epochResolutionCertified": False,
                "usedForConditionalBounds": True,
                "selectionReason": "allValidObservationsIncluded",
            })

            if previousTicks is not None and resolvedTicks < previousTicks:
                raise RuntimeError("Resolved timer moved backward; observations retained.")
            previousTicks = resolvedTicks

        # Center host arithmetic near the first observation to reduce cancellation.
        quantizationAllowance = 1 / candidateTicksPerSecond
        lowerBounds, upperBounds, midpointOffsets = [], [], []

        for observation in observations:
            deviceSeconds = (
                observation["candidateUnwrappedTicks"] - tickReference
            ) / candidateTicksPerSecond
            lowerBounds.append(
                (observation["hostBefore"] - hostReferenceBase)
                - deviceSeconds - quantizationAllowance
            )
            upperBounds.append(
                (observation["hostAfter"] - hostReferenceBase)
                - deviceSeconds + quantizationAllowance
            )
            midpointOffsets.append(
                (observation["hostMidpoint"] - hostReferenceBase) - deviceSeconds
            )

        lower = max(lowerBounds)
        upper = min(upperBounds)
        feasible = lower <= upper
        rtts = [observation["roundTripSeconds"] for observation in observations]

        last = observations[-1]
        tickDifference = (
            last["candidateUnwrappedTicks"] - first["candidateUnwrappedTicks"]
        )
        minimumElapsed = last["hostBefore"] - first["hostAfter"]
        maximumElapsed = last["hostAfter"] - first["hostBefore"]

        # Conditional first/last rate bounds, expanded by two ticks.
        rateBounds = None
        if minimumElapsed > 0 and tickDifference > 2:
            rateBounds = [
                (tickDifference - 2) / maximumElapsed,
                (tickDifference + 2) / minimumElapsed,
            ]

        summary = {
            "observationCount": len(observations),
            "selectedCountForConditionalBounds": len(observations),
            "hostMidpointSpanSeconds": last["hostMidpoint"] - first["hostMidpoint"],
            "rttSeconds": {
                "minimum": min(rtts),
                "median": percentile(rtts, 0.5),
                "p95": percentile(rtts, 0.95),
                "maximum": max(rtts),
            },
            "tickReference": tickReference,
            "hostReferenceBase": hostReferenceBase,
            "conditionalOffsetIntersection": {
                "feasible": feasible,
                "lowerRelativeToHostBase": lower,
                "upperRelativeToHostBase": upper,
                "emptyIntersectionSeconds": max(0.0, lower - upper),
                "midpointHostReference": (
                    hostReferenceBase + (lower + upper) / 2 if feasible else None
                ),
                "halfWidthSeconds": (upper - lower) / 2 if feasible else None,
                "nominalRateModelOnly": True,
                "totalWorstCaseAccuracyEstablished": False,
            },
            "midpointOffsetRangeSeconds": max(midpointOffsets) - min(midpointOffsets),
            "conditionalFirstLastRateBoundsHz": rateBounds,
            "candidateFrequencyInsideRateBounds": (
                rateBounds[0] <= candidateTicksPerSecond <= rateBounds[1]
                if rateBounds is not None else None
            ),
        }
        collection["summary"] = summary
        collection["status"] = (
            "conditionalNominalModelFeasible"
            if feasible else "inconsistentWithConditionalNominalModel"
        )

        boundText = (
            f"±{(upper - lower) * 0.5e6:.1f} us"
            if feasible else f"EMPTY by {(lower - upper) * 1e6:.1f} us"
        )
        rateText = (
            f"[{rateBounds[0]:.1f}, {rateBounds[1]:.1f}] Hz"
            if rateBounds is not None else "unavailable"
        )
        print(
            f"{collection['targetDurationSeconds'] * 1000:4.0f} ms "
            f"repeat {collection['repeatIndex']}: "
            f"actual={collection['actualDurationSeconds'] * 1000:.1f} ms, "
            f"n={len(observations)}, "
            f"RTT min/median/p95="
            f"{min(rtts) * 1e6:.1f}/"
            f"{percentile(rtts, 0.5) * 1e6:.1f}/"
            f"{percentile(rtts, 0.95) * 1e6:.1f} us, "
            f"conditional offset={boundText}, rate={rateText}",
            flush=True,
        )

    try:
        # Existing driver lock order; do not race its acquisition startup.
        with labJack._streamControlLock:
            if labJack.h is None:
                raise RuntimeError("LabJack is not connected.")
            if labJack.type != "T7":
                raise RuntimeError("This characterization targets the T7 only.")

            for thread in (labJack._digitalThread, labJack.acquisitionThread):
                if thread is not None and thread.is_alive():
                    raise RuntimeError("Acquisition worker is active; stop it explicitly first.")
            if labJack._digitalStreamActive or labJack.isReading:
                raise RuntimeError("Acquisition state is active; characterization refused.")

            with labJack._wordPulseLock:
                with labJack._digitalIOLock:
                    ljm = labJack.ljm
                    if ljm.eReadName(labJack.h, "STREAM_ENABLE") != 0:
                        raise RuntimeError("Device reports an active stream; no changes made.")

                    for registerName in ("CORE_TIMER", "STREAM_START_TIME_STAMP"):
                        address, dataType = ljm.nameToAddress(registerName)
                        report["registers"][registerName] = {
                            "addressFromInstalledLJM": int(address),
                            "dataTypeFromInstalledLJM": int(dataType),
                        }
                        print(
                            f"{registerName}: address={address}, LJM type={dataType}",
                            flush=True,
                        )

                    uint32Type = getattr(ljm.constants, "UINT32", None)
                    report["ljmUint32Type"] = uint32Type
                    if uint32Type is None:
                        raise RuntimeError("Installed LJM does not expose UINT32 for type checking.")
                    if report["registers"]["CORE_TIMER"]["dataTypeFromInstalledLJM"] != uint32Type:
                        raise RuntimeError("CORE_TIMER is not described as UINT32 by installed LJM.")

                    for repeatIndex in range(1, repeatCount + 1):
                        for targetDuration in durationTargets:
                            collection = {
                                "repeatIndex": repeatIndex,
                                "targetDurationSeconds": targetDuration,
                                "observations": [],
                                "status": "collecting",
                                "failure": None,
                            }
                            report["collections"].append(collection)
                            collectionStart = pglTimestamp.getSecs()

                            try:
                                while True:
                                    observation = {
                                        "hostBefore": pglTimestamp.getSecs(),
                                        "rawTicks": None,
                                        "usedForConditionalBounds": False,
                                        "selectionReason": "notYetAnalyzed",
                                    }
                                    collection["observations"].append(observation)

                                    try:
                                        rawValue = ljm.eReadName(labJack.h, "CORE_TIMER")
                                    finally:
                                        observation["hostAfter"] = pglTimestamp.getSecs()

                                    observation["rawValueRepresentation"] = repr(rawValue)
                                    numericValue = float(rawValue)
                                    if (
                                        isinstance(rawValue, bool)
                                        or not math.isfinite(numericValue)
                                        or numericValue != math.floor(numericValue)
                                        or not 0 <= numericValue < timerModulus
                                    ):
                                        raise RuntimeError(f"Invalid CORE_TIMER value: {rawValue!r}")

                                    observation.update({
                                        "rawTicks": int(numericValue),
                                        "roundTripSeconds": (
                                            observation["hostAfter"] - observation["hostBefore"]
                                        ),
                                        "hostMidpoint": (
                                            observation["hostBefore"]
                                            + (observation["hostAfter"] - observation["hostBefore"]) / 2
                                        ),
                                    })
                                    if observation["hostAfter"] - collectionStart >= targetDuration:
                                        break

                                collection["actualDurationSeconds"] = (
                                    pglTimestamp.getSecs() - collectionStart
                                )
                                summarizeCollection(collection)

                            except BaseException as error:
                                collection["status"] = "failed"
                                collection["failure"] = f"{type(error).__name__}: {error}"
                                raise

        report["status"] = "characterizationCompleted"

    except BaseException as error:
        report["status"] = "failedOrInterrupted"
        report["failure"] = f"{type(error).__name__}: {error}"
        raise

    finally:
        # A unique artifact is never overwritten. In-memory evidence remains
        # available even if saving fails.
        try:
            with artifactPath.open("x", encoding="utf-8") as artifactFile:
                json.dump(report, artifactFile, indent=2, allow_nan=False)
            print(f"\nRetained report: {artifactPath}", flush=True)
        except Exception as saveError:
            report["artifactSaveFailure"] = f"{type(saveError).__name__}: {saveError}"
            print(f"\nArtifact save failed: {saveError}", flush=True)

        print(
            "Full raw evidence remains in labJackClockCharacterizationRuns[-1].\n"
            "Caller-owned connection left open. No defaults selected.",
            flush=True,
        )

    return report

In [6]:
labJack = pglLabJack()

(pglLabJack) Opened T7 LabJack device via USB connection.
             serialNumber: 470023478 ipAddress: 0 port: 0 maxBytesPerMB: 64


In [ ]:
# Replace existingLabJack with the variable holding your open pglLabJack.
clockReport = characterizeLabJackClock(
    labJack,
    artifactDirectory=Path.home() / "pgl-clock-diagnostics",
)

CORE_TIMER: address=61520, LJM type=1
STREAM_START_TIME_STAMP: address=4026, LJM type=1
  50 ms repeat 1: actual=50.2 ms, n=170, RTT min/median/p95=167.2/255.0/526.2 us, conditional offset=±79.1 us, rate=[39693495.9, 40589478.6] Hz
 100 ms repeat 1: actual=100.0 ms, n=440, RTT min/median/p95=156.0/222.9/299.0 us, conditional offset=±73.8 us, rate=[39921351.0, 40081473.7] Hz
 250 ms repeat 1: actual=250.2 ms, n=1078, RTT min/median/p95=161.7/225.7/299.6 us, conditional offset=±70.8 us, rate=[39959513.4, 40066227.8] Hz
1000 ms repeat 1: actual=1000.2 ms, n=4185, RTT min/median/p95=125.2/232.7/302.5 us, conditional offset=±57.1 us, rate=[39987823.3, 40014769.1] Hz
  50 ms repeat 2: actual=50.0 ms, n=215, RTT min/median/p95=133.2/227.4/307.2 us, conditional offset=±61.1 us, rate=[39763750.5, 40252230.6] Hz
 100 ms repeat 2: actual=100.2 ms, n=415, RTT min/median/p95=166.2/237.2/312.8 us, conditional offset=±77.6 us, rate=[39906320.1, 40097954.5] Hz
 250 ms repeat 2: actual=250.2 ms, n=1091

In [10]:
import math
import numpy as np


def fitLabJackEndpointBrackets(observations, nominalTicksPerSecond=40_000_000):
    """
    Fit a positive affine device-to-host mapping constrained by ALL brackets.

    Required observation fields:
        unwrappedTicks, hostBefore, hostAfter

    Tick epochs must already have been resolved by the caller.

    Bounds are conditional on:
        - fresh register sampling during the read transaction;
        - one linear clock relationship over the observation interval;
        - the caller's epoch assignments;
        - the quantization allowance below.

    Host timestamp uncertainty and nonlinear within-run drift are not bounded.
    """
    if len(observations) < 3:
        raise ValueError("At least three observations are required.")

    tickReference = int(observations[0]["unwrappedTicks"])
    hostBase = float(observations[0]["hostBefore"])

    deviceSeconds = np.asarray([
        (int(row["unwrappedTicks"]) - tickReference) / nominalTicksPerSecond
        for row in observations
    ], dtype=float)
    hostBefore = np.asarray([
        float(row["hostBefore"]) - hostBase for row in observations
    ])
    hostAfter = np.asarray([
        float(row["hostAfter"]) - hostBase for row in observations
    ])

    if not (
        np.all(np.isfinite(deviceSeconds))
        and np.all(np.isfinite(hostBefore))
        and np.all(np.isfinite(hostAfter))
    ):
        raise ValueError("Nonfinite calibration coordinates.")
    if np.any(hostAfter < hostBefore):
        raise ValueError("A host bracket runs backward.")
    if np.any(np.diff(deviceSeconds) < 0):
        raise ValueError("Resolved device coordinates run backward.")
    if np.ptp(deviceSeconds) <= 0:
        raise ValueError("No device-time span.")

    # Conservative nominal one-tick allowance in host seconds.
    # Its applicability remains conditional on near-nominal operation.
    tickAllowance = 1 / nominalTicksPerSecond
    lower = hostBefore - tickAllowance
    upper = hostAfter + tickAllowance

    # Eliminate the offset from:
    #     lower[i] <= offset + scale*x[i] <= upper[i]
    #
    # For every pair:
    #     scale*(x[j] - x[i]) <= upper[j] - lower[i]
    #
    # This computes the feasible scale interval without excluding any row.
    scaleLower = 0.0
    scaleUpper = math.inf
    lowerSupport = None
    upperSupport = None

    for rowIndex in range(len(observations)):
        differences = deviceSeconds - deviceSeconds[rowIndex]
        limits = upper - lower[rowIndex]

        sameTime = differences == 0
        if np.any(limits[sameTime] < 0):
            raise ValueError("Inconsistent brackets at the same device coordinate.")

        positive = np.flatnonzero(differences > 0)
        if positive.size:
            candidates = limits[positive] / differences[positive]
            selected = int(np.argmin(candidates))
            candidate = float(candidates[selected])
            if candidate < scaleUpper:
                scaleUpper = candidate
                upperSupport = [rowIndex, int(positive[selected])]

        negative = np.flatnonzero(differences < 0)
        if negative.size:
            candidates = limits[negative] / differences[negative]
            selected = int(np.argmax(candidates))
            candidate = float(candidates[selected])
            if candidate > scaleLower:
                scaleLower = candidate
                lowerSupport = [rowIndex, int(negative[selected])]

    if not (
        math.isfinite(scaleLower)
        and math.isfinite(scaleUpper)
        and 0 < scaleLower <= scaleUpper
    ):
        raise ValueError(
            f"No adequately constrained positive slope: [{scaleLower}, {scaleUpper}]."
        )

    clockScale = scaleLower + (scaleUpper - scaleLower) / 2
    offsetLower = float(np.max(lower - clockScale * deviceSeconds))
    offsetUpper = float(np.min(upper - clockScale * deviceSeconds))
    if offsetLower > offsetUpper:
        raise ValueError("Empty offset intersection at the selected slope.")

    offset = offsetLower + (offsetUpper - offsetLower) / 2
    predicted = offset + clockScale * deviceSeconds
    midpoints = hostBefore + (hostAfter - hostBefore) / 2
    residuals = predicted - midpoints

    return {
        "method": "allBracketEndpointAffine-v1",
        "nominalTicksPerSecond": nominalTicksPerSecond,
        "tickReference": tickReference,
        "hostBase": hostBase,
        "hostReferenceOffset": offset,
        "hostReference": hostBase + offset,
        "clockScale": clockScale,
        "secondsPerTick": clockScale / nominalTicksPerSecond,
        "feasibleClockScale": [scaleLower, scaleUpper],
        "scaleLowerSupportIndices": lowerSupport,
        "scaleUpperSupportIndices": upperSupport,
        "offsetIntervalAtSelectedScale": [offsetLower, offsetUpper],
        "observationCount": len(observations),
        "deviceSpanSeconds": float(np.ptp(deviceSeconds)),
        "hostMidpointSpanSeconds": float(midpoints[-1] - midpoints[0]),
        "midpointResidualSeconds": residuals.tolist(),
        "midpointResidualRmsSeconds": float(np.sqrt(np.mean(residuals**2))),
        "quantizationAllowanceSeconds": tickAllowance,
        "linearModelRequired": True,
        "hostTimestampUncertainty": None,
        "withinRunNonlinearDriftBound": None,
        "totalWorstCaseAccuracyEstablished": False,
    }


def labJackEndpointHostInterval(mapping, observations, deviceTicks):
    """
    Conservative outer interval containing every feasible affine prediction.

    This is not necessarily the tightest projection of the feasible region.
    It does not bound departures from the assumed linear clock model.
    """
    nominalRate = mapping["nominalTicksPerSecond"]
    tickReference = mapping["tickReference"]
    hostBase = mapping["hostBase"]
    scaleLower, scaleUpper = mapping["feasibleClockScale"]
    allowance = mapping["quantizationAllowanceSeconds"]

    targetSeconds = (deviceTicks - tickReference) / nominalRate
    lowerBounds, upperBounds = [], []

    for row in observations:
        sourceSeconds = (row["unwrappedTicks"] - tickReference) / nominalRate
        delta = targetSeconds - sourceSeconds
        scaleTerms = (scaleLower * delta, scaleUpper * delta)
        lowerBounds.append(
            (row["hostBefore"] - hostBase) - allowance + min(scaleTerms)
        )
        upperBounds.append(
            (row["hostAfter"] - hostBase) + allowance + max(scaleTerms)
        )

    lower, upper = max(lowerBounds), min(upperBounds)
    if lower > upper:
        raise ValueError("Empty predicted host interval.")

    estimateRelative = (
        mapping["hostReferenceOffset"] + mapping["clockScale"] * targetSeconds
    )
    return {
        "hostTime": hostBase + estimateRelative,
        "conditionalHostInterval": [hostBase + lower, hostBase + upper],
        "conditionalErrorIntervalSeconds": [
            lower - estimateRelative,
            upper - estimateRelative,
        ],
        "intervalMethod": "conservativeAffineBracketProjection",
        "totalWorstCaseAccuracyEstablished": False,
    }


def testLabJackEndpointFitting():
    """HARDWARE-FREE tests. No device, files, or profiles are accessed."""
    nominalRate = 40_000_000

    # Test both fast and slow device clocks, with asymmetric host brackets.
    for trueScale in (0.99998, 1.00002):
        tickReference = 100_000
        hostReference = 1200.0
        rows = []

        for deviceSeconds in (0.0, 0.02, 0.04, 600.0, 600.02, 600.04):
            ticks = tickReference + round(deviceSeconds * nominalRate)
            trueHost = hostReference + trueScale * (
                (ticks - tickReference) / nominalRate
            )
            rows.append({
                "unwrappedTicks": ticks,
                "hostBefore": trueHost - 30e-6,
                "hostAfter": trueHost + 110e-6,
            })

        mapping = fitLabJackEndpointBrackets(rows)
        lower, upper = mapping["feasibleClockScale"]
        assert lower <= trueScale <= upper

        midpointTicks = tickReference + 300 * nominalRate
        result = labJackEndpointHostInterval(mapping, rows, midpointTicks)
        trueMidpoint = hostReference + trueScale * 300
        lowerHost, upperHost = result["conditionalHostInterval"]
        assert lowerHost <= trueMidpoint <= upperHost

        # Repeating identical evidence must not shrink a common clock bound.
        duplicatedMapping = fitLabJackEndpointBrackets(rows + [])
        duplicatedResult = labJackEndpointHostInterval(
            duplicatedMapping, rows + [], midpointTicks
        )
        assert result == duplicatedResult

        repeatedRows = [
            dict(row) for row in rows for repeatIndex in range(3)
        ]
        repeatedMapping = fitLabJackEndpointBrackets(repeatedRows)
        repeatedResult = labJackEndpointHostInterval(
            repeatedMapping, repeatedRows, midpointTicks
        )
        assert np.allclose(
            repeatedResult["conditionalErrorIntervalSeconds"],
            result["conditionalErrorIntervalSeconds"],
            rtol=0, atol=1e-12,
        )

    # Contradictory observations must fail, not be silently discarded.
    contradictoryRows = [
        {"unwrappedTicks": 0, "hostBefore": 0.0, "hostAfter": 0.0001},
        {"unwrappedTicks": nominalRate, "hostBefore": 1.0, "hostAfter": 1.0001},
        {"unwrappedTicks": nominalRate, "hostBefore": 1.01, "hostAfter": 1.0101},
    ]
    try:
        fitLabJackEndpointBrackets(contradictoryRows)
    except ValueError:
        pass
    else:
        raise AssertionError("Contradictory brackets were accepted.")

    # No temporal leverage must fail.
    try:
        fitLabJackEndpointBrackets([
            {"unwrappedTicks": 1, "hostBefore": 0.0, "hostAfter": 0.001}
            for repeatIndex in range(3)
        ])
    except ValueError:
        pass
    else:
        raise AssertionError("Zero-span observations were accepted.")

    print("Hardware-free endpoint-fit tests passed.")


testLabJackEndpointFitting()

Hardware-free endpoint-fit tests passed.


In [14]:
# HARDWARE-FREE: verify the actual event serialization contract.
# No hardware, file writes, saved profiles, or production modifications.

import json
from copy import deepcopy

from pgl.pglEvent import pglEvent, pglEventDigitalInput


def testLabJackTimingSerialization():
    calibrationSnapshot = {
        "calibrationId": "synthetic-beginning-calibration",
        "mappingId": "synthetic-live-mapping",
        "nominalTicksPerSecond": 40_000_000,
        "tickReference": (1 << 32) * 3 + 12345,
        "hostReference": 1200.0,
        "clockScale": 1.0,
        "observations": [{
            "hostBefore": 1199.9999,
            "hostAfter": 1200.0001,
            "rawTicks": 12345,
            "unwrappedTicks": (1 << 32) * 3 + 12345,
            "selected": True,
            "selectionReason": "syntheticTestObservation",
        }],
        "uncertainty": {
            "conditionalReferenceErrorSeconds": [-0.0001, 0.0001],
            "rateUncertainty": None,
            "withinRunNonlinearDriftBound": None,
            "totalWorstCaseAccuracyEstablished": False,
        },
    }

    event = pglEventDigitalInput(
        channel="CIO1",
        eventType="rising",
        timestamp=1200.00007,
        sampleIndex=7,
        scanRate=100000.0,
        deviceType="LabJack",
        timestampMethod="syntheticCalibratedDeviceSample",
    )
    event.acquisitionId = "synthetic-acquisition"
    event.segmentId = 2
    event.calibrationId = calibrationSnapshot["calibrationId"]
    event.mappingId = calibrationSnapshot["mappingId"]
    event.deviceTimestamp = {
        "method": "streamStartStampPlusSampleOffset",
        "rawStreamStartTicks": 12345,
        "unwrappedStreamStartTicks": (1 << 32) * 3 + 12345,
        "sampleOffsetTicks": 2800.0,
        "hardwareCapturedEdge": False,
    }
    event.isEstimated = False
    event.clockCalibrationSnapshot = deepcopy(calibrationSnapshot)

    encoded = event.toJSON(strict=True)
    restored = pglEventDigitalInput.fromJSON(encoded, strict=True)

    assert isinstance(restored, pglEventDigitalInput)
    assert json.loads(restored.toJSON(strict=True)) == json.loads(encoded)
    assert restored.sampleTime == 7 / 100000.0
    assert restored.deviceTimestamp["unwrappedStreamStartTicks"] == (
        (1 << 32) * 3 + 12345
    )
    assert restored.clockCalibrationSnapshot["uncertainty"]["rateUncertainty"] is None

    # A restored snapshot must be independent of the original object.
    restored.clockCalibrationSnapshot["observations"][0]["selected"] = False
    assert event.clockCalibrationSnapshot["observations"][0]["selected"] is True

    # Diagnostic timestamps remain host times; absent device time stays None.
    notification = pglEvent(type="digitalInputRecovery")
    notification.timestamp = 1210.5
    notification.timestampMethod = "hostDiagnosticNotification"
    notification.deviceTimestamp = None
    notification.supportingObservation = deepcopy(event.deviceTimestamp)

    restoredNotification = pglEvent.fromJSON(
        notification.toJSON(strict=True), strict=True
    )
    assert restoredNotification.timestamp == 1210.5
    assert restoredNotification.deviceTimestamp is None
    assert restoredNotification.supportingObservation == event.deviceTimestamp

    # Legacy events must not acquire fabricated calibration metadata on load.
    legacyEvent = pglEventDigitalInput(
        channel="CIO1",
        eventType="rising",
        timestamp=100.0,
        sampleIndex=10,
        scanRate=1000.0,
        deviceType="LabJack",
        timestampMethod="streamStartMidpointEstimate",
    )
    restoredLegacy = pglEventDigitalInput.fromJSON(
        legacyEvent.toJSON(strict=True), strict=True
    )
    assert restoredLegacy.timestampMethod == "streamStartMidpointEstimate"
    assert not hasattr(restoredLegacy, "mappingId")
    assert not hasattr(restoredLegacy, "deviceTimestamp")

    # Strict timing snapshots must reject nonfinite mapping parameters.
    invalidEvent = deepcopy(event)
    invalidEvent.clockCalibrationSnapshot["clockScale"] = float("nan")
    try:
        invalidEvent.toJSON(strict=True)
    except ValueError:
        pass
    else:
        raise AssertionError("Strict serialization accepted a nonfinite clock scale.")

    print("PASS: digital-event timing provenance survives strict JSON round-trip.")
    print("PASS: snapshots are independent; unavailable values remain None.")
    print("PASS: host diagnostic timestamps and legacy timing remain unchanged.")
    print("PASS: nonfinite clock parameters are rejected.")
    print("No hardware accessed, files written, or production code changed.")


testLabJackTimingSerialization()

PASS: digital-event timing provenance survives strict JSON round-trip.
PASS: snapshots are independent; unavailable values remain None.
PASS: host diagnostic timestamps and legacy timing remain unchanged.
PASS: nonfinite clock parameters are rejected.
No hardware accessed, files written, or production code changed.
